In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import streamlit
import requests

print("✅ All libraries loaded successfully!")

✅ All libraries loaded successfully!


In [2]:
import requests

# Kitwe coordinates
lat = -12.8024
lon = 28.2132

# Correct parameter names for RE community
url = (
    "https://power.larc.nasa.gov/api/temporal/daily/point?"
    "parameters=ALLSKY_SFC_SW_DWN,RH2M,T2M,CLOUD_AMT"
    "&community=RE"
    f"&longitude={lon}"
    f"&latitude={lat}"
    "&start=20200101"
    "&end=20251231"
    "&format=CSV"
)

print("Fetching data from NASA POWER...")
response = requests.get(url, timeout=120)

with open("../data/nasa_power_kitwe_raw.csv", "wb") as f:
    f.write(response.content)

print("✅ Done!")
print(f"File size: {len(response.content) / 1024:.1f} KB")

Fetching data from NASA POWER...


ConnectTimeout: HTTPSConnectionPool(host='power.larc.nasa.gov', port=443): Max retries exceeded with url: /api/temporal/daily/point?parameters=ALLSKY_SFC_SW_DWN,RH2M,T2M,CLOUD_AMT&community=RE&longitude=28.2132&latitude=-12.8024&start=20200101&end=20251231&format=CSV (Caused by ConnectTimeoutError(<HTTPSConnection(host='power.larc.nasa.gov', port=443) at 0x245e032cb10>, 'Connection to power.larc.nasa.gov timed out. (connect timeout=120)'))

In [ ]:
with open("../data/nasa_power_kitwe_raw.csv", "r") as f:
    print(f.read())

-BEGIN HEADER-
NASA/POWER Source Native Resolution Daily Data 
Dates (month/day/year): 01/01/2020 through 12/31/2025 in LST
Location: latitude  -12.8024   longitude 28.2132 
elevation from MERRA-2: Average for 0.5 x 0.625 degree lat/lon region = 1231.59 meters
The value for missing source data that cannot be computed or is outside of the sources availability range: -999 
parameter(s): 
ALLSKY_SFC_SW_DWN     CERES SYN1deg All Sky Surface Shortwave Downward Irradiance (kW-hr/m^2/day) 
RH2M                  MERRA-2 Relative Humidity at 2 Meters (%) 
T2M                   MERRA-2 Temperature at 2 Meters (C) 
CLOUD_AMT             CERES SYN1deg Cloud Amount (%) 
-END HEADER-
YEAR,MO,DY,ALLSKY_SFC_SW_DWN,RH2M,T2M,CLOUD_AMT
2020,1,1,4.932,83.52,22.36,93.38
2020,1,2,4.5946,82.25,22.79,96.34
2020,1,3,6.7109,85.36,22.21,74.3
2020,1,4,5.3273,81.26,22.7,97.51
2020,1,5,4.9447,83.3,21.57,99.74
2020,1,6,6.5522,78.05,22.15,85.75
2020,1,7,6.017,85.6,21.6,92.72
2020,1,8,5.7206,83.12,22.04,84.63
2020,1,9

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../data/nasa_power_kitwe_raw.csv",
    skiprows=12,
    na_values=-999
)

df.columns = ['YEAR', 'MON', 'DAY', 'GHI', 'RH2M', 'T2M', 'CLOUD_AMT']
df = df.dropna()

# Create DOY - pandas needs specific column names
df['DATE'] = pd.to_datetime({
    'year': df['YEAR'],
    'month': df['MON'],
    'day': df['DAY']
})
df['DOY'] = df['DATE'].dt.dayofyear

print(f"Total rows: {df.shape[0]}")
print(df[['YEAR', 'MON', 'DAY', 'DOY', 'GHI', 'RH2M', 'T2M', 'CLOUD_AMT']].head())

Total rows: 2192
   YEAR  MON  DAY  DOY     GHI   RH2M    T2M  CLOUD_AMT
0  2020    1    1    1  4.9320  83.52  22.36      93.38
1  2020    1    2    2  4.5946  82.25  22.79      96.34
2  2020    1    3    3  6.7109  85.36  22.21      74.30
3  2020    1    4    4  5.3273  81.26  22.70      97.51
4  2020    1    5    5  4.9447  83.30  21.57      99.74


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[['T2M', 'RH2M', 'CLOUD_AMT']]
y = df['GHI']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print("✅ Data split and scaled successfully!")

Training samples: 1534
Testing samples: 658
✅ Data split and scaled successfully!


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("=== Dataset Statistics ===")
print(df[['GHI', 'T2M', 'RH2M', 'CLOUD_AMT']].describe())

=== Dataset Statistics ===
               GHI          T2M         RH2M    CLOUD_AMT
count  2192.000000  2192.000000  2192.000000  2192.000000
mean      5.681827    21.282578    65.685109    49.561478
std       1.023673     3.302669    18.678434    36.860622
min       1.800200    12.820000    20.560000     0.040000
25%       5.098000    19.167500    51.187500    13.177500
50%       5.743800    21.380000    70.495000    45.525000
75%       6.351400    23.082500    82.092500    88.812500
max       8.251700    31.400000    93.290000    99.940000


In [ ]:
plt.figure(figsize=(8, 6))
correlation = df[['GHI', 'T2M', 'RH2M', 'CLOUD_AMT']].corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap - Kitwe Solar Irradiance')
plt.tight_layout()
plt.savefig('../data/correlation_heatmap.png')
plt.show()
print("✅ Heatmap saved!")

✅ Heatmap saved!


C:\Users\Norah Chisha\AppData\Local\Temp\ipykernel_22848\1038010343.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df['GHI'].hist(ax=axes[0,0], bins=30, color='orange')
axes[0,0].set_title('GHI Distribution')
df['T2M'].hist(ax=axes[0,1], bins=30, color='red')
axes[0,1].set_title('Temperature Distribution')
df['RH2M'].hist(ax=axes[1,0], bins=30, color='blue')
axes[1,0].set_title('Humidity Distribution')
df['CLOUD_AMT'].hist(ax=axes[1,1], bins=30, color='gray')
axes[1,1].set_title('Cloud Cover Distribution')
plt.tight_layout()
plt.savefig('../data/distributions.png')
plt.show()
print("✅ Distributions saved!")

✅ Distributions saved!


C:\Users\Norah Chisha\AppData\Local\Temp\ipykernel_22848\2391777987.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].scatter(df['T2M'], df['GHI'], alpha=0.3, color='red')
axes[0].set_xlabel('Temperature (T2M)')
axes[0].set_ylabel('GHI')
axes[0].set_title('Temperature vs GHI')

axes[1].scatter(df['RH2M'], df['GHI'], alpha=0.3, color='blue')
axes[1].set_xlabel('Relative Humidity (RH2M)')
axes[1].set_ylabel('GHI')
axes[1].set_title('Humidity vs GHI')

axes[2].scatter(df['CLOUD_AMT'], df['GHI'], alpha=0.3, color='gray')
axes[2].set_xlabel('Cloud Cover (CLOUD_AMT)')
axes[2].set_ylabel('GHI')
axes[2].set_title('Cloud Cover vs GHI')

plt.tight_layout()
plt.savefig('../data/scatter_plots.png')
plt.show()
print("✅ Scatter plots saved!")

✅ Scatter plots saved!


C:\Users\Norah Chisha\AppData\Local\Temp\ipykernel_22848\3142529498.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"R² Score:  {r2:.4f}")
print(f"MSE:       {mse:.4f}")
print(f"RMSE:      {rmse:.4f}")

print(f"\nModel Coefficients:")
for feature, coef in zip(['T2M', 'RH2M', 'CLOUD_AMT'], model.coef_):
    print(f"  {feature}: {coef:.4f}")
print(f"  Intercept: {model.intercept_:.4f}")

R² Score:  0.5597
MSE:       0.4595
RMSE:      0.6779

Model Coefficients:
  T2M: 0.7666
  RH2M: 0.2943
  CLOUD_AMT: -0.9541
  Intercept: 5.7010


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

# Evaluation metrics
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(y_test - y_pred))

print("=== Model Evaluation Results ===")
print(f"R² Score:  {r2:.4f}")
print(f"MSE:       {mse:.4f}")
print(f"RMSE:      {rmse:.4f}")
print(f"MAE:       {mae:.4f}")

# Actual vs Predicted plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.3, color='blue')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual GHI')
plt.ylabel('Predicted GHI')
plt.title('Actual vs Predicted GHI')
plt.legend()
plt.tight_layout()
plt.savefig('../data/actual_vs_predicted.png')
plt.show()
print("✅ Evaluation plot saved!")

=== Model Evaluation Results ===
R² Score:  0.5597
MSE:       0.4595
RMSE:      0.6779
MAE:       0.5261
✅ Evaluation plot saved!


C:\Users\Norah Chisha\AppData\Local\Temp\ipykernel_22848\1026561771.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Calculate Kitwe-specific GHI percentiles
low_threshold = df['GHI'].quantile(0.33)
high_threshold = df['GHI'].quantile(0.67)

print(f"Kitwe GHI Statistics:")
print(f"Minimum:          {df['GHI'].min():.4f} kWh/m²/day")
print(f"Maximum:          {df['GHI'].max():.4f} kWh/m²/day")
print(f"Mean:             {df['GHI'].mean():.4f} kWh/m²/day")
print(f"Median:           {df['GHI'].median():.4f} kWh/m²/day")
print(f"33rd percentile:  {low_threshold:.4f} kWh/m²/day")
print(f"67th percentile:  {high_threshold:.4f} kWh/m²/day")

Kitwe GHI Statistics:
Minimum:          1.8002 kWh/m²/day
Maximum:          8.2517 kWh/m²/day
Mean:             5.6818 kWh/m²/day
Median:           5.7438 kWh/m²/day
33rd percentile:  5.3325 kWh/m²/day
67th percentile:  6.1238 kWh/m²/day


In [ ]:
import joblib

# Save the trained model
joblib.dump(model, '../models/mlr_model.joblib')

# Save the scaler too — very important!
joblib.dump(scaler, '../models/scaler.joblib')

print("✅ Model saved to models/mlr_model.joblib")
print("✅ Scaler saved to models/scaler.joblib")

✅ Model saved to models/mlr_model.joblib
✅ Scaler saved to models/scaler.joblib
